In [2]:
import pandas as pd 
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np
import geopandas as gpd
from pyproj import datadir
import rioxarray

import xarray as xr
datadir.set_data_dir("/home/jupyter-daniela/.conda/envs/peru_environment/share/proj")



In [3]:
shapefile_peru = Path("/home/jupyter-daniela/suyana/geometries/areas_pesca_peru/areas_pesca_peru.shp")
base_path = Path("/home/jupyter-daniela/suyana/peru_production/features/")

gdf_peru = gpd.read_file(shapefile_peru)

In [4]:

mapa_regiones = {
    'TUMBES': 'norte',
    'PIURA': 'norte',
    'LAMBAYEQUE': 'norte',
    'LA LIBERTAD': 'centro',
    'ANCASH': 'centro',
    'LIMA': 'centro',
    'ICA': 'centro',
    'AREQUIPA': 'sur',
    'MOQUEGUA': 'sur',
    'TACNA': 'sur'
}
gdf_peru["region_macro"] = gdf_peru["DPTO"].map(mapa_regiones)

In [26]:
from shapely.geometry import Point
from tqdm import tqdm

registros = []
hycom_sources = Path("/home/jupyter-daniela/suyana/sources/hycom")

# Obtener todos los archivos
all_files = sorted(hycom_sources.rglob("hycom_*.nc"))
print(f"Total archivos encontrados: {len(all_files)}\n")

# Procesar cada archivo
for archivo in tqdm(all_files, desc="Procesando archivos HYCOM"):
    try:
        ds = xr.open_dataset(archivo)
        
        # Verificar variables requeridas
        if "water_temp" not in ds or "salinity" not in ds or "time" not in ds:
            continue
        
        # Resample a diario (3-hourly → daily)
        ds_daily = ds.resample(time="1D").mean(dim="time", skipna=True)
        
        # Procesar por departamento
        for _, row in gdf_peru.iterrows():
            dpto = row["DPTO"]
            region = row["region_macro"]
            geom = row.geometry
            
            # Crear máscara booleana de puntos dentro del departamento
            lats = ds_daily['lat'].values
            lons = ds_daily['lon'].values
            mask = np.zeros((len(lats), len(lons)), dtype=bool)
            
            for j, lon in enumerate(lons):
                for i, lat in enumerate(lats):
                    point = Point(lon, lat)
                    if geom.contains(point):
                        mask[i, j] = True
            
            if mask.sum() == 0:  # Sin datos en este departamento
                continue
            
            # Extraer datos para puntos dentro de la máscara
            temp_in_geom = ds_daily["water_temp"].values[:, :, mask]  # (time, depth, num_points)
            salt_in_geom = ds_daily["salinity"].values[:, :, mask]
            
            # Media sobre los puntos válidos
            temp_mean = np.nanmean(temp_in_geom, axis=2)[:, 0]  # (time,)
            salt_mean = np.nanmean(salt_in_geom, axis=2)[:, 0]
            
            # Crear DataFrame
            df_merge = pd.DataFrame({
                "fecha": ds_daily['time'].values,
                "temperatura": temp_mean,
                "salinidad": salt_mean,
                "DPTO": dpto,
                "region_macro": region
            })
            registros.append(df_merge)
    
    except Exception as e:
        print(f"Error en {archivo.name}: {e}")
        continue

# Concatenar todos los registros
print(f"\n✅ Total registros recolectados: {sum(len(r) for r in registros)}")
if registros:
    df_final = pd.concat(registros, ignore_index=True)
    df_final['fecha'] = pd.to_datetime(df_final['fecha'])
    print(f"DataFrame final shape: {df_final.shape}")
else:
    print("❌ No se recolectaron registros")


Procesando 2015: 12 archivos...

Procesando 2016: 12 archivos...

Procesando 2016: 12 archivos...

Procesando 2017: 12 archivos...

Procesando 2017: 12 archivos...

Procesando 2018: 12 archivos...

Procesando 2018: 12 archivos...

Procesando 2019: 12 archivos...

Procesando 2019: 12 archivos...


KeyboardInterrupt: 

In [20]:
df_final = df_final[["fecha", "temperatura", "salinidad", "DPTO", "region_macro"]]

In [21]:
df_final = df_final.copy()
df_final.loc[:, "anio"] = pd.to_datetime(df_final["fecha"]).dt.year
df_final.loc[:, "dia_juliano"] = pd.to_datetime(df_final["fecha"]).dt.dayofyear

clim = (
    df_final.groupby(["region_macro", "dia_juliano"])[["temperatura", "salinidad"]]
    .mean()
    .rename(columns={"temperatura": "temp_clim", "salinidad": "sal_clim"})
    .reset_index()
)

df_final = df_final.merge(clim, on=["region_macro", "dia_juliano"], how="left")

df_final.loc[:, "anom_temp"] = df_final["temperatura"] - df_final["temp_clim"]
df_final.loc[:, "anom_sal"] = df_final["salinidad"] - df_final["sal_clim"]
df_final.loc[:, "anom_sal_ref35"] = df_final["salinidad"] - 35.1


In [22]:
output_path = Path("/home/jupyter-daniela/suyana/peru_production/outputs/")

df_final.to_csv(output_path / "salinity_temperature_peru_daily.csv", index=False)